In [4]:
import sys

In [7]:
import sys
print(sys.executable)


c:\Users\inesc\Downloads\ISCTE\Planapp\Planapp\.venv\Scripts\python.exe


In [8]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [11]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install nltk
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\inesc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
import sklearn


In [13]:
# --- Environment Setup for VS Code ---

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# Download NLTK stopwords (only the first time)
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords_pt = stopwords.words('portuguese')




[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\inesc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#Sprint 2

In [19]:
# carregar o dataset 

parquet_path = "../../data/bronze/cision_news_20251110.parquet"

sample = pd.read_parquet(parquet_path, engine="pyarrow")

sample.head()


,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,116477697,Isabel Ferreira reafirma compromisso com Braga...,A candidata à Presidência da Câmara Municipal ...,No link available,2025-04-03T00:00:00+00:00,Imprensa,sem autor,139.4,http://pt.cision.com/cp2013/clippingdetails.as...,None
1,116477659,Legislativas 25 - Júlia Rodrigues escolhida pa...,"A presidente da Câmara de Mirandela, Júlia Rod...",No link available,2025-04-03T00:00:00+00:00,Imprensa,"Fernando Pires, António Gonçalves Rodrigues",267.2,http://pt.cision.com/cp2013/clippingdetails.as...,None
2,116456100,Alargamento dos passeios na envolvente à Casa-...,"Arrancou na passada segunda-feira, o alargamen...",No link available,2025-04-02T00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,None
3,116477524,Macedo de Cavaleiros - Câmara investiu dois mi...,A Câmara de Macedo de Cavaleiros investiu mais...,No link available,2025-04-03T00:00:00+00:00,Imprensa,Glória Lopes,180.1,http://pt.cision.com/cp2013/clippingdetails.as...,None
4,116477522,OPINIÃO - Clarificação,Obviamente que é imperativo que a atuação de q...,No link available,2025-04-03T00:00:00+00:00,Imprensa,Leite Mário José,133.6,http://pt.cision.com/cp2013/clippingdetails.as...,None


In [20]:
sample.head(10)

,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,116477697,Isabel Ferreira reafirma compromisso com Braga...,A candidata à Presidência da Câmara Municipal ...,No link available,2025-04-03T00:00:00+00:00,Imprensa,sem autor,139.4,http://pt.cision.com/cp2013/clippingdetails.as...,None
1,116477659,Legislativas 25 - Júlia Rodrigues escolhida pa...,"A presidente da Câmara de Mirandela, Júlia Rod...",No link available,2025-04-03T00:00:00+00:00,Imprensa,"Fernando Pires, António Gonçalves Rodrigues",267.2,http://pt.cision.com/cp2013/clippingdetails.as...,None
2,116456100,Alargamento dos passeios na envolvente à Casa-...,"Arrancou na passada segunda-feira, o alargamen...",No link available,2025-04-02T00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,None
3,116477524,Macedo de Cavaleiros - Câmara investiu dois mi...,A Câmara de Macedo de Cavaleiros investiu mais...,No link available,2025-04-03T00:00:00+00:00,Imprensa,Glória Lopes,180.1,http://pt.cision.com/cp2013/clippingdetails.as...,None
4,116477522,OPINIÃO - Clarificação,Obviamente que é imperativo que a atuação de q...,No link available,2025-04-03T00:00:00+00:00,Imprensa,Leite Mário José,133.6,http://pt.cision.com/cp2013/clippingdetails.as...,None
5,116477197,O atraso na Variante das Palhagueiras é o refl...,Há mais de 20 anos que é reconhecido pelos tor...,No link available,2025-04-04T00:00:00+00:00,Imprensa,João Paulo Reis,279.1,http://pt.cision.com/cp2013/clippingdetails.as...,None
6,116455753,António Soares é o candidato do Bloco de Esque...,António Soares é o candidato do Bloco de Esque...,No link available,2025-03-30T00:00:00+00:00,Imprensa,sem autor,365.9,http://pt.cision.com/cp2013/clippingdetails.as...,None
7,116455708,Fernando Vale e Andreia Neto candidatos a depu...,Fernando Vale e Andreia Neto são a escolha da ...,No link available,2025-03-30T00:00:00+00:00,Imprensa,sem autor,108.1,http://pt.cision.com/cp2013/clippingdetails.as...,None
8,116476980,Região - Politécnico de Leiria aponta três cam...,Politécnico aponta três caminhos para o desenv...,No link available,2025-04-03T00:00:00+00:00,Imprensa,Elisabete Cruz,623.7,http://pt.cision.com/cp2013/clippingdetails.as...,None
9,116466403,José Luís Carneiro desvaloriza polémica em Bra...,Legislativas 2025\n\nO antigo candidato à lide...,https://ominho.pt/jose-luis-carneiro-desvalori...,2025-04-02T00:00:00+00:00,Internet,sem autor,7154.0,http://pt.cision.com/cp2013/clippingdetails.as...,None


In [21]:
print(sample.shape)
print(sample.columns)
display(sample.head())
display(sample.dtypes)
print(sample.isna().sum())
print("Duplicados por corpo de noticia:", sample.duplicated(subset=["noticia"]).sum()) #mas têm valores unitarios dif


(60645, 10)
Index(['id', 'titulo', 'noticia', 'link', 'data_publicacao', 'tipo_meio',
       'autores', 'valor_publicitario', 'guid', 'categoria'],
      dtype='object')


,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,116477697,Isabel Ferreira reafirma compromisso com Braga...,A candidata à Presidência da Câmara Municipal ...,No link available,2025-04-03T00:00:00+00:00,Imprensa,sem autor,139.4,http://pt.cision.com/cp2013/clippingdetails.as...,None
1,116477659,Legislativas 25 - Júlia Rodrigues escolhida pa...,"A presidente da Câmara de Mirandela, Júlia Rod...",No link available,2025-04-03T00:00:00+00:00,Imprensa,"Fernando Pires, António Gonçalves Rodrigues",267.2,http://pt.cision.com/cp2013/clippingdetails.as...,None
2,116456100,Alargamento dos passeios na envolvente à Casa-...,"Arrancou na passada segunda-feira, o alargamen...",No link available,2025-04-02T00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,None
3,116477524,Macedo de Cavaleiros - Câmara investiu dois mi...,A Câmara de Macedo de Cavaleiros investiu mais...,No link available,2025-04-03T00:00:00+00:00,Imprensa,Glória Lopes,180.1,http://pt.cision.com/cp2013/clippingdetails.as...,None
4,116477522,OPINIÃO - Clarificação,Obviamente que é imperativo que a atuação de q...,No link available,2025-04-03T00:00:00+00:00,Imprensa,Leite Mário José,133.6,http://pt.cision.com/cp2013/clippingdetails.as...,None


id                     object
titulo                 object
noticia                object
link                   object
data_publicacao        object
tipo_meio              object
autores                object
valor_publicitario    float64
guid                   object
categoria              object
dtype: object

id                        0
titulo                    0
noticia                   0
link                      0
data_publicacao           0
tipo_meio                 0
autores                   0
valor_publicitario        0
guid                      0
categoria             60645
dtype: int64
Duplicados por corpo de noticia: 723


só a coluna "categoria" é que tem nulos

In [22]:
# Datas e valores numéricos
sample["data_publicacao"] = pd.to_datetime(sample["data_publicacao"], errors="coerce")
sample["valor_publicitario"] = pd.to_numeric(sample["valor_publicitario"], errors="coerce")

# Normalizar texto das colunas relevantes
def norm_txt(s):
    s = str(s).lower()
    s = re.sub(r"[\:\;\,\.\!\?\(\)\[\]\-\–\—\"\']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

#vamos ver com o corpo da noticia
sample["noticia"] = sample["noticia"].map(norm_txt)


##TF-IDF (melhor opção de todas)

In [8]:
#4. Vetorização TF-IDF + matriz de similaridade
vectorizer = TfidfVectorizer(stop_words=list(stopwords_pt), ngram_range=(1,2), max_features=30000, min_df=3)
X = vectorizer.fit_transform(sample["noticia"])

# Similaridade (mais baixo aumenta grupos)
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components

k = 25           # começa com 25
threshold = 0.76 # o mesmo que já usavas

nbrs = NearestNeighbors(n_neighbors=k, metric="cosine", n_jobs=-1)
nbrs.fit(X)

# grafo de distâncias (esparso) -> similaridade
G = nbrs.kneighbors_graph(mode="distance").tocsr()
G.data = 1.0 - G.data
G.data[G.data < threshold] = 0.0
G.eliminate_zeros()

# grupos por componentes conexos (labels = grupo de cada linha)
B = G.copy(); B.data[:] = 1.0
n_comp, labels = connected_components(csgraph=B, directed=False, return_labels=True)


In [9]:
#5. Agrupamento por similaridade
sample = sample.reset_index(drop=True)
sample["grupo_id"] = pd.Series(labels).astype("Int64")


In [10]:
# para contagens
from collections import Counter

sizes = Counter(sample["grupo_id"].value_counts().tolist())
print("Distribuição de tamanhos de grupo:", sizes)
print("Grupos totais:", sample["grupo_id"].nunique())
print("Grupos com 1 notícia:", (sample["grupo_id"].value_counts()==1).sum())



Distribuição de tamanhos de grupo: Counter({1: 36107, 2: 4032, 3: 1041, 4: 513, 5: 268, 6: 170, 7: 100, 8: 97, 9: 70, 11: 47, 10: 47, 12: 33, 14: 31, 13: 27, 17: 22, 15: 22, 19: 17, 21: 12, 18: 11, 16: 11, 25: 9, 20: 8, 26: 5, 24: 5, 29: 4, 34: 3, 32: 3, 30: 3, 22: 3, 37: 2, 35: 2, 31: 2, 27: 2, 81: 1, 73: 1, 70: 1, 49: 1, 47: 1, 44: 1, 41: 1, 40: 1, 38: 1, 28: 1, 23: 1})
Grupos totais: 42740
Grupos com 1 notícia: 36107


In [15]:
# Tabela com número de notícias por grupo
contagem_por_grupo = (
    sample.groupby("grupo_id")
          .size()
          .reset_index(name="num_noticias")
)

# Lista com os ids dos grupos que têm exatamente x notícias p.e.
grupos_x = contagem_por_grupo.loc[
    contagem_por_grupo["num_noticias"] ==47,"grupo_id"
].tolist()

print("Grupos com x notícias:", grupos_x)
print("Total de grupos com x notícias:", len(grupos_x))

Grupos com x notícias: [17699]
Total de grupos com x notícias: 1


In [16]:
#6. Escolher o líder por valor publicitário

def lider_valor_publicitario(df, grupo, col_valor="valor_publicitario"):
    # apenas retorna o índice da notícia com maior valor_publicitario dentro do grupo
    idx_validos = [int(i) for i in grupo if 0 <= int(i) < len(df)]
    return int(df.loc[idx_validos, col_valor].idxmax())

leaders = (sample.groupby("grupo_id")["valor_publicitario"].idxmax().dropna().astype(int).tolist())
sample["is_lider"] = False
sample.loc[leaders, "is_lider"] = True

In [17]:
#dos 4 grupos que tem 15 noticias
gid = grupos_x[0] #0 é o 1 grupo, 1 o segundo e etc
exemplo_4 = sample.loc[sample["grupo_id"] == gid, ["grupo_id","is_lider","noticia","data_publicacao", "valor_publicitario"]].head(50)
display(exemplo_4)


,grupo_id,is_lider,noticia,data_publicacao,valor_publicitario
24108,17699,False,ministro da educação fernando alexandre falou ...,2025-09-06 00:00:00+00:00,34795.20
24322,17699,False,o reitor da universidade do porto u porto antó...,2025-09-06 00:00:00+00:00,1307.44
24333,17699,True,reitor da universidade do porto denunciou ter ...,2025-09-06 00:00:00+00:00,48181.25
24338,17699,False,o ministro da educação ciência e inovação diss...,2025-09-06 00:00:00+00:00,9428.44
24340,17699,False,o reitor da universidade do porto u porto antó...,2025-09-06 00:00:00+00:00,15636.52
25970,17699,False,fernando alexandre diz que ainda não falou com...,2025-09-06 00:00:00+00:00,29355.52
25994,17699,False,o ministro da educação ciência e inovação diss...,2025-09-06 00:00:00+00:00,21583.91
26020,17699,False,“não ainda não falei com o reitor da u porto e...,2025-09-06 00:00:00+00:00,1500.00
26022,17699,False,o ministro da educação ciência e inovação diss...,2025-09-06 00:00:00+00:00,34065.94
26023,17699,False,o ministro da educação ciência e inovação diss...,2025-09-06 00:00:00+00:00,28443.46


In [18]:
ver_carac = sample[sample["grupo_id"] == 391 ]

display(ver_carac[["is_lider", "noticia", "tipo_meio", "valor_publicitario"]])


,is_lider,noticia,tipo_meio,valor_publicitario
448,True,os produtores portugueses têm de se preparar p...,Internet,46003.0
613,False,sector do calçado português anuncia que não de...,Internet,10935.0


In [19]:
# Inspecionar um grupo grande
gid_ex = sample["grupo_id"].value_counts().idxmax()
display(sample[sample["grupo_id"] == gid_ex][["id", "is_lider", "noticia", "valor_publicitario"]].head(50))


,id,is_lider,noticia,valor_publicitario
85,116472699,False,plan il portugal s caretaker ad government wil...,100.0
1262,117171220,False,dams il a visit to mér tola alentejo by the mi...,637.6
2213,116592436,False,by michael bruxo michael bruxo@port portugalre...,336.2
2531,116527050,False,ps pips psd to post in presenting electoral pr...,772.0
4773,116905942,False,were facing a huge challenge here in portugal ...,750.0
6171,116716811,False,olháo il portugal s environment and energy min...,428.9
6266,116765324,False,te renewal of the citizen card can now be done...,127.6
6275,116764337,False,portugal is showing strong signs of economic r...,452.4
8106,116971746,False,18 000 undocumented immigrants not wanted in p...,750.0
9279,117048750,False,first 4 574 immigrants lold portugal will not ...,1414.2
